In [1]:
from pyspark.sql import SparkSession

In [2]:
from pyspark.sql import SparkSession

# Initialize local Spark Session with security configurations
spark = SparkSession.builder \
    .appName("NYCtaxiDataAnalysisproject") \
    .config("spark.driver.memory", "4g") \
    .config("spark.hadoop.security.authentication", "simple") \
    .config("spark.hadoop.security.authorization", "false") \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 17:44:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, LongType

# Define explicit schema based on the NYC TLC data dictionary
yellow_taxi_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    # Note: passenger_count and RatecodeID are semantically integers, but frequently
    # stored as Double/Float in TLC parquet files due to upstream NaN handling.
    StructField("passenger_count", DoubleType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", DoubleType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True),
    StructField("cbd_congestion_fee", DoubleType(), True)  # Enacted Jan 2025
])

In [4]:
# Install required packages if needed
import subprocess
import sys

for package in ["pyarrow", "pandas", "numpy"]:
    try:
        __import__(package)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import pyarrow.parquet as pq
import numpy as np

# Read the file with PyArrow (no Hadoop/Spark issues)
print("Loading parquet file with PyArrow...")
table = pq.read_table("/Users/avikumart/Documents/GitHub/DA-DS-Questions/PySpark/yellow_tripdata_2026-04.parquet")

# Display schema
print("\nParquet Schema:")
print(table.schema)

# Convert directly to Spark DataFrame using PyArrow Table
# This bypasses pandas entirely
print("\nConverting to Spark DataFrame...")
df = spark.createDataFrame(table.to_pylist())

# Verify the result
print(f"✓ Successfully loaded {df.count()} rows")
print("\nDataFrame Schema:")
df.printSchema()

# Show sample data
print("\nFirst 5 rows:")
df.show(5)


Loading parquet file with PyArrow...

Parquet Schema:
VendorID: int32
tpep_pickup_datetime: timestamp[us]
tpep_dropoff_datetime: timestamp[us]
passenger_count: int64
trip_distance: double
RatecodeID: int64
store_and_fwd_flag: large_string
PULocationID: int32
DOLocationID: int32
payment_type: int64
fare_amount: double
extra: double
mta_tax: double
tip_amount: double
tolls_amount: double
improvement_surcharge: double
total_amount: double
congestion_surcharge: double
Airport_fee: double
cbd_congestion_fee: double

Converting to Spark DataFrame...


26/06/11 17:45:42 WARN FileSystem: Cannot load filesystem
java.util.ServiceConfigurationError: org.apache.hadoop.fs.FileSystem: Provider org.apache.hadoop.fs.viewfs.ViewFileSystem could not be instantiated
	at java.base/java.util.ServiceLoader.fail(ServiceLoader.java:552)
	at java.base/java.util.ServiceLoader$ProviderImpl.newInstance(ServiceLoader.java:712)
	at java.base/java.util.ServiceLoader$ProviderImpl.get(ServiceLoader.java:672)
	at java.base/java.util.ServiceLoader$2.next(ServiceLoader.java:1256)
	at org.apache.hadoop.fs.FileSystem.loadFileSystems(FileSystem.java:3525)
	at org.apache.hadoop.fs.FileSystem.getFileSystemClass(FileSystem.java:3562)
	at org.apache.hadoop.fs.FsUrlStreamHandlerFactory.<init>(FsUrlStreamHandlerFactory.java:77)
	at org.apache.spark.sql.internal.SharedState$.liftedTree2$1(SharedState.scala:209)
	at org.apache.spark.sql.internal.SharedState$.org$apache$spark$sql$internal$SharedState$$setFsUrlStreamHandlerFactory(SharedState.scala:208)
	at org.apache.spark.

✓ Successfully loaded 3831240 rows

DataFrame Schema:
root
 |-- Airport_fee: double (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- VendorID: long (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)


First 5 rows:
+-----------+----------

Traceback (most recent call last):
  File "/Users/avikumart/Documents/GitHub/DA-DS-Questions/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/avikumart/Documents/GitHub/DA-DS-Questions/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


In [5]:
# write a groupy query to find the average fare amount by payment type
average_fare_by_payment_type = df.groupBy("payment_type").avg("fare_amount")
average_fare_by_payment_type.show()

26/06/11 17:50:17 WARN TaskSetManager: Stage 4 contains a task of very large size (62651 KiB). The maximum recommended task size is 1000 KiB.


+------------+------------------+
|payment_type|  avg(fare_amount)|
+------------+------------------+
|           1|20.050289348491678|
|           3| 10.50504324966079|
|           2|19.415355891355926|
|           4|  3.02163714111179|
|           0|25.917277946858484|
+------------+------------------+



In [6]:
# find the average fare amount by payment type and passenger count
average_fare_by_payment_and_passenger = df.groupBy("payment_type", "passenger_count").avg("fare_amount")
average_fare_by_payment_and_passenger.show()

26/06/11 17:50:24 WARN TaskSetManager: Stage 7 contains a task of very large size (62651 KiB). The maximum recommended task size is 1000 KiB.


+------------+---------------+------------------+
|payment_type|passenger_count|  avg(fare_amount)|
+------------+---------------+------------------+
|           3|              0|19.920735294117645|
|           3|              5|1.6600000000000001|
|           4|              0|13.777283950617285|
|           2|              6|19.296347826086954|
|           1|              3|22.637795012035525|
|           2|              5|17.305137614678905|
|           1|              0|  16.9709006368139|
|           1|              1|19.540116944653825|
|           3|              4| 7.942898550724639|
|           1|              4| 28.27103728248232|
|           4|              5| 6.746153846153846|
|           4|              3|1.3346820809248556|
|           4|              4|1.3983959537572253|
|           3|              3| 7.151322751322752|
|           4|              2|1.0867285464098073|
|           1|              5| 17.16063845153836|
|           2|              3|23.498945164667052|


In [7]:
# find the average trip count by each day of the month
average_trip_count_by_day = df.groupBy("tpep_pickup_datetime").count()
average_trip_count_by_day.show()

26/06/11 17:50:28 WARN TaskSetManager: Stage 10 contains a task of very large size (62651 KiB). The maximum recommended task size is 1000 KiB.


+--------------------+-----+
|tpep_pickup_datetime|count|
+--------------------+-----+
| 2026-04-01 00:44:27|    1|
| 2026-04-01 00:41:48|    1|
| 2026-04-01 01:46:02|    1|
| 2026-04-01 02:13:24|    1|
| 2026-04-01 04:06:13|    2|
| 2026-04-01 05:50:11|    2|
| 2026-04-01 06:05:37|    2|
| 2026-04-01 06:41:59|    2|
| 2026-04-01 06:15:17|    2|
| 2026-04-01 07:22:07|    3|
| 2026-04-01 07:26:23|    1|
| 2026-04-01 08:49:14|    2|
| 2026-04-01 08:38:26|    6|
| 2026-04-01 08:55:03|    2|
| 2026-04-01 01:27:07|    1|
| 2026-04-01 03:13:16|    1|
| 2026-04-01 05:03:21|    1|
| 2026-04-01 06:30:08|    2|
| 2026-04-01 07:42:50|    3|
| 2026-04-01 07:51:35|    1|
+--------------------+-----+
only showing top 20 rows


In [8]:
# filter the anomolies and engineer time based features
from pyspark.sql.functions import col, hour, dayofweek, month, expr

# Filter out anomalies based on fare amount and trip distance
filtered_df = df.filter((col("fare_amount") > 0) & (col("trip_distance") > 0)) \
    .withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_dayofweek", dayofweek(col("tpep_pickup_datetime"))) \
    .withColumn("pickup_month", month(col("tpep_pickup_datetime")))

In [ ]:
filtered_df.printSchema()

root
 |-- Airport_fee: double (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- VendorID: long (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_dayofweek: integer (nullable = true)


In [11]:
filtered_df.show(5)

+-----------+------------+------------+----------+--------+------------------+--------------------+-----+-----------+---------------------+-------+---------------+------------+------------------+----------+------------+------------+---------------------+--------------------+-------------+-----------+----------------+------------+
|Airport_fee|DOLocationID|PULocationID|RatecodeID|VendorID|cbd_congestion_fee|congestion_surcharge|extra|fare_amount|improvement_surcharge|mta_tax|passenger_count|payment_type|store_and_fwd_flag|tip_amount|tolls_amount|total_amount|tpep_dropoff_datetime|tpep_pickup_datetime|trip_distance|pickup_hour|pickup_dayofweek|pickup_month|
+-----------+------------+------------+----------+--------+------------------+--------------------+-----+-----------+---------------------+-------+---------------+------------+------------------+----------+------------+------------+---------------------+--------------------+-------------+-----------+----------------+------------+
|   

26/06/11 17:52:52 WARN TaskSetManager: Stage 14 contains a task of very large size (62651 KiB). The maximum recommended task size is 1000 KiB.
Traceback (most recent call last):
  File "/Users/avikumart/Documents/GitHub/DA-DS-Questions/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/avikumart/Documents/GitHub/DA-DS-Questions/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


In [12]:
# fare amount by each hour and passenger count
average_fare_by_hour_and_passenger = filtered_df.groupBy("pickup_hour", "passenger_count").avg("fare_amount")
average_fare_by_hour_and_passenger.show()

26/06/11 17:53:58 WARN TaskSetManager: Stage 15 contains a task of very large size (62651 KiB). The maximum recommended task size is 1000 KiB.


+-----------+---------------+------------------+
|pickup_hour|passenger_count|  avg(fare_amount)|
+-----------+---------------+------------------+
|         11|              0|16.523700305810397|
|         15|              6|23.445804195804193|
|         19|              3|20.274615951372272|
|          0|              4|21.514496800465388|
|         12|              4|28.638752775721684|
|         14|              6|20.215328467153284|
|         15|              2|24.536037177510114|
|         22|              1|19.465079880698145|
|         12|              2| 22.75961114496205|
|          8|              6|17.124884792626727|
|         16|              3| 24.76967232202262|
|          6|              5|19.335294117647063|
|          3|              3|18.962218749999998|
|         12|              3|22.900261077844306|
|         21|              2|19.146605348804304|
|          1|              0|15.523456790123456|
|         22|              6|17.910859728506786|
|          5|       

In [16]:
from pyspark.sql.functions import avg, sum, rank
from pyspark.sql.window import Window  

# hourly stats
hourly_stats = filtered_df.groupBy("pickup_hour") \
    .agg(avg("fare_amount").alias("avg_fare"), sum("passenger_count").alias("total_passengers")) \
    .orderBy("pickup_hour")

In [17]:
hourly_stats.show()

26/06/11 17:55:25 WARN TaskSetManager: Stage 18 contains a task of very large size (62651 KiB). The maximum recommended task size is 1000 KiB.


+-----------+------------------+----------------+
|pickup_hour|          avg_fare|total_passengers|
+-----------+------------------+----------------+
|          0| 22.46359292332691|           89782|
|          1| 19.99159070337845|           56168|
|          2| 19.03050065637599|           35629|
|          3|20.547553813490154|           23183|
|          4|25.041691932117967|           15986|
|          5|27.579614895376263|           20573|
|          6| 24.31704044203022|           44699|
|          7|21.861862524132913|           90990|
|          8|21.145855194457475|          130041|
|          9|20.261529097152817|          151086|
|         10| 20.25195579118506|          168142|
|         11|20.794284548802292|          186976|
|         12| 20.80295004546972|          201424|
|         13| 21.42048145221012|          211152|
|         14|22.229074593509964|          229119|
|         15|22.176392554310265|          241457|
|         16| 21.72395921630987|          247712|


In [18]:
# using window functinos to find the top 3 hours with the highest average fare
window_spec = Window.orderBy(col("avg_fare").desc())
top_hours_by_avg_fare = hourly_stats.withColumn("rank", rank().over(window_spec)).filter(col("rank") <= 3)
top_hours_by_avg_fare.show()

26/06/11 17:58:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/11 17:58:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/11 17:58:01 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/11 17:58:01 WARN TaskSetManager: Stage 21 contains a task of very large size (62651 KiB). The maximum recommended task size is 1000 KiB.
26/06/11 17:58:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/11 17:58:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-----------+------------------+----------------+----+
|pickup_hour|          avg_fare|total_passengers|rank|
+-----------+------------------+----------------+----+
|          5|27.579614895376263|           20573|   1|
|          4|25.041691932117967|           15986|   2|
|          6| 24.31704044203022|           44699|   3|
+-----------+------------------+----------------+----+



26/06/11 17:58:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/11 17:58:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/11 17:58:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/11 17:58:05 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [19]:
# Window Function: Find top 3 longest trips for each hour of the day
window_spec = Window.partitionBy("pickup_hour").orderBy(col("trip_distance").desc())
top_trips_by_hour = filtered_df.withColumn("rank", rank().over(window_spec)).filter(col("rank") <= 3)
top_trips_by_hour.select("pickup_hour", "trip_distance", "fare_amount", "passenger_count").show()

26/06/11 17:59:52 WARN TaskSetManager: Stage 33 contains a task of very large size (62651 KiB). The maximum recommended task size is 1000 KiB.


+-----------+-------------+-----------+---------------+
|pickup_hour|trip_distance|fare_amount|passenger_count|
+-----------+-------------+-----------+---------------+
|          0|    201712.74|      46.19|           NULL|
|          0|       210.82|      300.0|              1|
|          0|       112.78|      371.2|           NULL|
|          1|    206279.09|      27.85|           NULL|
|          1|    145720.36|       22.2|           NULL|
|          1|         95.0|      280.0|              1|
|          2|        171.1|     1110.4|              3|
|          2|       139.24|      300.0|              1|
|          2|        50.11|      180.4|           NULL|
|          3|        79.29|      140.0|              1|
|          3|        66.26|      417.4|              1|
|          3|        48.31|      331.3|              1|
|          4|         64.9|      392.2|              1|
|          4|        64.51|      341.8|              1|
|          4|        61.71|      263.4|         

In [20]:
# feature engineering: create a new column for trip duration in minutes
filtered_df = filtered_df.withColumn("trip_duration_minutes", (col("tpep_dropoff_datetime").cast("long") - col("tpep_pickup_datetime").cast("long")) / 60)
filtered_df.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_duration_minutes").show(5)

26/06/11 18:12:19 WARN TaskSetManager: Stage 36 contains a task of very large size (62651 KiB). The maximum recommended task size is 1000 KiB.


+--------------------+---------------------+---------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_duration_minutes|
+--------------------+---------------------+---------------------+
| 2026-04-01 00:40:05|  2026-04-01 00:52:44|                12.65|
| 2026-04-01 00:09:19|  2026-04-01 00:21:29|   12.166666666666666|
| 2026-04-01 00:15:29|  2026-04-01 00:34:14|                18.75|
| 2026-04-01 00:14:20|  2026-04-01 00:27:49|   13.483333333333333|
| 2026-04-01 00:04:53|  2026-04-01 00:11:54|    7.016666666666667|
+--------------------+---------------------+---------------------+
only showing top 5 rows


Traceback (most recent call last):
  File "/Users/avikumart/Documents/GitHub/DA-DS-Questions/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/avikumart/Documents/GitHub/DA-DS-Questions/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


In [21]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml import Pipeline


# create a feature vector for machine learning
feature_columns = ["trip_distance", "passenger_count", "pickup_hour", "pickup_dayofweek", "pickup_month"]
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")    

# create the indexer from the vendor id column
indexer = StringIndexer(inputCol="VendorID", outputCol="vendor_index")

In [22]:
# build the pipeline
pipeline = Pipeline(stages=[indexer, assembler])
model = pipeline.fit(filtered_df)
transformed_df = model.transform(filtered_df)
transformed_df.select("features", "vendor_index").show(5, truncate=False)

26/06/11 18:15:57 WARN TaskSetManager: Stage 37 contains a task of very large size (62651 KiB). The maximum recommended task size is 1000 KiB.


+----------------------+------------+
|features              |vendor_index|
+----------------------+------------+
|[2.8,1.0,0.0,4.0,4.0] |1.0         |
|[7.37,1.0,0.0,4.0,4.0]|0.0         |
|[7.66,1.0,0.0,4.0,4.0]|0.0         |
|[7.9,0.0,0.0,4.0,4.0] |1.0         |
|[1.34,1.0,0.0,4.0,4.0]|0.0         |
+----------------------+------------+
only showing top 5 rows


26/06/11 18:16:01 WARN TaskSetManager: Stage 43 contains a task of very large size (62651 KiB). The maximum recommended task size is 1000 KiB.
Traceback (most recent call last):
  File "/Users/avikumart/Documents/GitHub/DA-DS-Questions/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/avikumart/Documents/GitHub/DA-DS-Questions/.venv/lib/python3.11/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


In [23]:
# stop the Spark session
spark.stop()